In [51]:
%pip install psycopg[binary]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
import psycopg

In [53]:
from psycopg import sql

In [54]:
%pip install random

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement random (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for random


In [55]:
import random
import string
import pandas as pd


def generate_site_code():
    letters = "".join(random.choices(string.ascii_uppercase, k=3))
    numbers = "".join(random.choices(string.digits, k=3))
    return letters + numbers


def generate_site_data(count):
    data = []
    used_site_codes = set()
    used_coordinates = set()

    while len(data) < count:

        site_code = generate_site_code()
        latitude = round(random.uniform(-90, 90), 2)
        longitude = round(random.uniform(-180, 180), 2)

        # Skip duplicate site codes
        if site_code in used_site_codes:
            continue

        # Skip duplicate coordinates
        if (latitude, longitude) in used_coordinates:
            continue

        used_site_codes.add(site_code)
        used_coordinates.add((latitude, longitude))

        data.append(
            {
                "site_code": site_code,
                "latitude": latitude,
                "longitude": longitude,
            }
        )

    return pd.DataFrame(data)

In [56]:
def create_db_meta(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [57]:
create_db_meta("meta")

Database 'meta' created successfully!


In [58]:
def create_table_meta():
    try:
        with psycopg.connect(
            dbname="meta",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS metadata (
                        site_name VARCHAR(100) Not NULL,
                        latitude DOUBLE PRECISION Not NULL,
                        longitude DOUBLE PRECISION Not NULL
                    );
                """)

            conn.commit()
            print("metadata table created.")

    except psycopg.Error as e:
        print(e)

In [59]:
create_table_meta()

metadata table created.


In [60]:
def insert_sites():

    sites_df = generate_site_data(7000)

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:
        with conn.cursor() as cur:
            for _, row in sites_df.iterrows():
                cur.execute(
                    """
                    INSERT INTO metadata
                    (site_name, latitude, longitude)
                    VALUES (%s, %s, %s)                                                                         
                    """,
                    (row["site_code"], row["latitude"], row["longitude"]),
                )
        conn.commit()

    print("Sites inserted successfully!")

In [61]:
insert_sites()

Sites inserted successfully!


In [62]:
def get_sites():

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:

        with conn.cursor() as cur:
            cur.execute(""" 
                SELECT site_name, latitude, longitude
                FROM metadata
            """)

            sites = cur.fetchall()

    return sites

In [63]:
data = list(get_sites())

In [64]:
type(data[0])

tuple

In [65]:
len(data)

7000

In [66]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [67]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [68]:
def create_db_site_weather(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [69]:
create_db_site_weather("site_weather")

Database 'site_weather' created successfully!


In [70]:
import requests
from retry_requests import retry
import openmeteo_requests

session = requests.Session()

retry_session = retry(session, retries=5, backoff_factor=0.2)

openmeteo = openmeteo_requests.Client(session=retry_session)

In [ ]:
import time
import datetime

import psycopg
import requests
import openmeteo_requests

# ============================================================
# SETTINGS
# ============================================================

MAX_LOCATIONS = 7000
BATCH_SIZE = 10

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

# How long to wait before trying a rate-limited request again.
# We do NOT reset Open-Meteo's quota ourselves.
RATE_LIMIT_WAIT_SECONDS = 65 * 60


# ============================================================
# DATABASE SETTINGS
# ============================================================

DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"


# ============================================================
# GET SITES FROM meta.metadata
# ============================================================


def get_sites_from_metadata():
    """
    Read the real sites stored in the meta database.

    Returns:
        list of tuples:
        (site_name, latitude, longitude)
    """

    with psycopg.connect(
        dbname="meta",
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cursor:

            cursor.execute(
                """
                SELECT
                    site_name,
                    latitude,
                    longitude
                FROM metadata
                WHERE latitude IS NOT NULL
                  AND longitude IS NOT NULL
                ORDER BY site_name
                LIMIT %s;
            """,
                (MAX_LOCATIONS,),
            )

            sites = cursor.fetchall()

    return sites


# ============================================================
# MAIN WEATHER COLLECTION
# ============================================================

try:

    # ========================================================
    # Get sites from metadata database
    # ========================================================

    data = get_sites_from_metadata()

    print("=" * 70)
    print("Sites loaded from meta.metadata")
    print("=" * 70)

    print(f"Total sites loaded: {len(data)}")

    if len(data) < MAX_LOCATIONS:

        raise RuntimeError(
            f"metadata contains only {len(data)} sites. "
            f"{MAX_LOCATIONS} sites are required."
        )

    # Make sure we process exactly 7000 sites.

    data = data[:MAX_LOCATIONS]

    print(f"Sites selected for weather collection: {len(data)}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Expected successful batches: " f"{MAX_LOCATIONS // BATCH_SIZE}")

    print("=" * 70)

    # ========================================================
    # PostgreSQL weather database
    # ========================================================

    with psycopg.connect(
        dbname="site_weather",
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cursor:

            # =================================================
            # Create weather table
            # =================================================

            cursor.execute("""
                CREATE TABLE IF NOT EXISTS site_weather (
                    site_name VARCHAR(100) NOT NULL,
                    time_interval TIMESTAMPTZ NOT NULL,
                    temperature REAL NOT NULL,
                    humidity REAL NOT NULL,
                    solar_radiance REAL NOT NULL,

                    UNIQUE (
                        site_name,
                        time_interval
                    )
                );
            """)

            conn.commit()

            print("\nsite_weather table ready.")

            # =================================================
            # Open-Meteo HTTP session
            # =================================================

            session = requests.Session()

            openmeteo = openmeteo_requests.Client(session=session)

            # =================================================
            # Counters
            # =================================================

            locations_processed = 0

            successful_batches = 0

            api_requests = 0

            # =================================================
            # Process exactly 10 locations per request
            # =================================================

            for batch_start in range(0, MAX_LOCATIONS, BATCH_SIZE):

                batch_end = min(batch_start + BATCH_SIZE, MAX_LOCATIONS)

                batch = data[batch_start:batch_end]

                # ------------------------------------------------
                # This should always be 10 except the final batch.
                # Since 7000 is divisible by 10, final batch is 10.
                # ------------------------------------------------

                if len(batch) != BATCH_SIZE:

                    raise RuntimeError(f"Unexpected batch size: " f"{len(batch)}")

                # =================================================
                # Extract site information
                # =================================================

                site_codes = [row[0] for row in batch]

                latitudes = [float(row[1]) for row in batch]

                longitudes = [float(row[2]) for row in batch]

                batch_number = (batch_start // BATCH_SIZE) + 1

                # =================================================
                # Open-Meteo parameters
                # =================================================

                params = {
                    # 10 latitude values
                    "latitude": latitudes,
                    # 10 longitude values
                    "longitude": longitudes,
                    # Weather variables
                    "hourly": [
                        "temperature_2m",
                        "relative_humidity_2m",
                        "direct_radiation",
                    ],
                    # Requested period
                    "start_date": START_DATE,
                    "end_date": END_DATE,
                }

                # =================================================
                # Send ONE request for these 10 locations
                # =================================================

                while True:

                    try:

                        api_requests += 1

                        print("\n" + "=" * 70)

                        print(f"Batch {batch_number}/700")

                        print(f"API HTTP request #{api_requests}")

                        print(
                            f"Sending locations "
                            f"{batch_start + 1}"
                            f"-"
                            f"{batch_end}"
                        )

                        print(f"Number of locations in request: " f"{len(batch)}")

                        print(f"First site: {site_codes[0]}")

                        print(f"Last site: {site_codes[-1]}")

                        print("=" * 70)

                        # =================================================
                        # ONE HTTP REQUEST
                        # =================================================

                        responses = openmeteo.weather_api(
                            OPEN_METEO_URL, params=params, method="POST"
                        )

                        # =================================================
                        # Verify that Open-Meteo returned
                        # all 10 locations
                        # =================================================

                        if len(responses) != len(batch):

                            raise RuntimeError(
                                f"Expected "
                                f"{len(batch)} responses, "
                                f"but Open-Meteo returned "
                                f"{len(responses)}."
                            )

                        print(
                            f"SUCCESS: "
                            f"1 HTTP request returned "
                            f"{len(responses)} locations."
                        )

                        # =================================================
                        # Request successful
                        # =================================================

                        break

                    except Exception as e:

                        error_message = str(e)

                        print("\nOpen-Meteo error:")
                        print(error_message)

                        # =================================================
                        # Hourly API limit
                        # =================================================

                        if "Hourly API request limit exceeded" in error_message:

                            print("\nHourly API limit reached.")

                            print("The quota cannot be reset " "by the client.")

                            print(
                                f"Waiting "
                                f"{RATE_LIMIT_WAIT_SECONDS / 60:.0f} "
                                f"minutes before retrying "
                                f"the SAME batch..."
                            )

                            time.sleep(RATE_LIMIT_WAIT_SECONDS)

                            # IMPORTANT:
                            # Do NOT increment batch number.
                            # Do NOT move to the next batch.
                            #
                            # The same 10 sites will be retried.

                            continue

                        # =================================================
                        # Minute API limit
                        # =================================================

                        elif "Minutely API request limit exceeded" in error_message:

                            print("\nMinute API limit reached.")

                            print("Waiting 65 seconds...")

                            time.sleep(65)

                            continue

                        # =================================================
                        # Other error
                        # =================================================

                        else:

                            raise

                # =================================================
                # Process the 10 returned locations
                # =================================================

                batch_locations_saved = 0

                for site_code, response in zip(site_codes, responses):

                    # ------------------------------------------------
                    # Get hourly object
                    # ------------------------------------------------

                    hourly = response.Hourly()

                    if hourly is None:

                        raise RuntimeError(
                            f"No hourly data returned " f"for site {site_code}"
                        )

                    # ------------------------------------------------
                    # Weather arrays
                    # ------------------------------------------------

                    temperatures = hourly.Variables(0).ValuesAsNumpy()

                    humidity = hourly.Variables(1).ValuesAsNumpy()

                    radiation = hourly.Variables(2).ValuesAsNumpy()

                    # ------------------------------------------------
                    # Make sure all three arrays have
                    # the same number of records
                    # ------------------------------------------------

                    number_of_hours = len(temperatures)

                    if (
                        len(humidity) != number_of_hours
                        or len(radiation) != number_of_hours
                    ):

                        raise RuntimeError(
                            f"Hourly array length mismatch " f"for site {site_code}"
                        )

                    # ------------------------------------------------
                    # API timestamp
                    # ------------------------------------------------

                    start_timestamp = hourly.Time()

                    start_time = datetime.datetime.fromtimestamp(
                        start_timestamp, tz=datetime.timezone.utc
                    )

                    # ------------------------------------------------
                    # Use API-provided interval
                    # ------------------------------------------------

                    interval_seconds = hourly.Interval()

                    # =================================================
                    # Store every hourly record
                    # =================================================

                    for j in range(number_of_hours):

                        time_interval = start_time + datetime.timedelta(
                            seconds=(interval_seconds * j)
                        )

                        cursor.execute(
                            """
                            INSERT INTO site_weather (
                                site_name,
                                time_interval,
                                temperature,
                                humidity,
                                solar_radiance
                            )
                            VALUES (
                                %s,
                                %s,
                                %s,
                                %s,
                                %s
                            )
                            ON CONFLICT (
                                site_name,
                                time_interval
                            )
                            DO NOTHING;
                            """,
                            (
                                site_code,
                                time_interval,
                                float(temperatures[j]),
                                float(humidity[j]),
                                float(radiation[j]),
                            ),
                        )

                    # ------------------------------------------------
                    # Count successful location
                    # ------------------------------------------------

                    batch_locations_saved += 1

                    print(
                        f"Saved: " f"{site_code} " f"({number_of_hours} hourly records)"
                    )

                # =================================================
                # Commit AFTER THE WHOLE BATCH
                # =================================================

                conn.commit()

                # =================================================
                # Update counters
                # =================================================

                locations_processed += batch_locations_saved

                successful_batches += 1

                print("\n" + "-" * 70)

                print(f"Batch {batch_number}/700 " f"SUCCESS")

                print(f"Locations saved in batch: " f"{batch_locations_saved}/10")

                print(f"Total locations saved: " f"{locations_processed}/7000")

                print(f"Successful batches: " f"{successful_batches}/700")

                print(f"HTTP requests attempted: " f"{api_requests}")

                print("-" * 70)

            # ====================================================
            # Final verification
            # ====================================================

            print("\n")
            print("=" * 70)
            print("FINAL RESULT")
            print("=" * 70)

            print(f"Locations processed: " f"{locations_processed}/7000")

            print(f"Successful batches: " f"{successful_batches}/700")

            print(f"HTTP requests attempted: " f"{api_requests}")

            # ====================================================
            # Count database rows
            # ====================================================

            cursor.execute("""
                SELECT COUNT(*)
                FROM site_weather;
            """)

            total_rows = cursor.fetchone()[0]

            print(f"Total weather rows in database: " f"{total_rows}")

            # ====================================================
            # Expected number of hourly rows
            # ====================================================

            print("\nExpected rows depend on the requested " "date interval.")

            # ====================================================
            # Verify every requested site exists
            # ====================================================

            cursor.execute("""
                SELECT COUNT(DISTINCT site_name)
                FROM site_weather;
            """)

            distinct_sites = cursor.fetchone()[0]

            print(f"Distinct sites in weather table: " f"{distinct_sites}")

            print("=" * 70)


except psycopg.Error as e:

    print("\nPostgreSQL error:")
    print(e)


except Exception as e:

    print("\nProgram error:")
    print(e)

site_weather table ready.

API request #1
Sending 10 locations
Locations: 1 - 10
Batch 0: 10 locations returned
Location 1/7000 saved: MUU670
Location 2/7000 saved: EOP568
Location 3/7000 saved: QIM604
Location 4/7000 saved: KZV666
Location 5/7000 saved: BEJ610
Location 6/7000 saved: OIB723
Location 7/7000 saved: EIY246
Location 8/7000 saved: FKK659
Location 9/7000 saved: WWO367
Location 10/7000 saved: FYO742

API request #2
Sending 10 locations
Locations: 11 - 20
Batch 10: 10 locations returned
Location 11/7000 saved: POU855
Location 12/7000 saved: MEX180
Location 13/7000 saved: MAQ103
Location 14/7000 saved: FIN774
Location 15/7000 saved: TIW183
Location 16/7000 saved: STO486
Location 17/7000 saved: CWA558
Location 18/7000 saved: YFM549
Location 19/7000 saved: NKA670
Location 20/7000 saved: RKQ842

API request #3
Sending 10 locations
Locations: 21 - 30
Batch 20: 10 locations returned
Location 21/7000 saved: MQV638
Location 22/7000 saved: SUS859
Location 23/7000 saved: LRK087
Location

In [ ]:
# url = "https://api.open-meteo.com/v1/forecast"

# import time

# MAX_LOCATIONS = 7000
# BATCH_SIZE = 30

# locations_processed = 0

# for i in range(0, len(data), 30):

#     batch = data[i : i + 30]

#     site_codes = [row[0] for row in batch]
#     latitudes = [row[1] for row in batch]
#     longitudes = [row[2] for row in batch]

#     params = {
#         "latitude": latitudes,
#         "longitude": longitudes,
#         "hourly": ["temperature_2m", "relative_humidity_2m", "direct_radiation"],
#     }

#     while True:
#         try:
#             responses = openmeteo.weather_api(url, params=params, method="POST")

#             print("Batch:", i, "Responses:", len(responses))
#             break  # successful request, while loop se bahar

#         except Exception as e:

#             if "Minutely API request limit exceeded" in str(e):

#                 print("API limit reached. Waiting 60 seconds...")
#                 time.sleep(60)

#             else:
#                 raise e

Batch: 0 Responses: 30
Batch: 30 Responses: 30
Batch: 60 Responses: 30
Batch: 90 Responses: 30
Batch: 120 Responses: 30
Batch: 150 Responses: 30
Batch: 180 Responses: 30
Batch: 210 Responses: 30
Batch: 240 Responses: 30
Batch: 270 Responses: 30
Batch: 300 Responses: 30
Batch: 330 Responses: 30
Batch: 360 Responses: 30
Batch: 390 Responses: 30
Batch: 420 Responses: 30
Batch: 450 Responses: 30
Batch: 480 Responses: 30
Batch: 510 Responses: 30
Batch: 540 Responses: 30
Batch: 570 Responses: 30
API limit reached. Waiting 60 seconds...
Batch: 600 Responses: 30
Batch: 630 Responses: 30
Batch: 660 Responses: 30
Batch: 690 Responses: 30
Batch: 720 Responses: 30
Batch: 750 Responses: 30
Batch: 780 Responses: 30
Batch: 810 Responses: 30
Batch: 840 Responses: 30
Batch: 870 Responses: 30
Batch: 900 Responses: 30
Batch: 930 Responses: 30
Batch: 960 Responses: 30
Batch: 990 Responses: 30
Batch: 1020 Responses: 30
Batch: 1050 Responses: 30
Batch: 1080 Responses: 30
Batch: 1110 Responses: 30
Batch: 114

KeyboardInterrupt: 

In [ ]:
# len(responses)

500

In [ ]:
# for site_code, response in zip(site_codes, responses):

#     hourly = response.Hourly()

#     temperature = hourly.Variables(0).ValuesAsNumpy()
#     humidity = hourly.Variables(1).ValuesAsNumpy()
#     radiation = hourly.Variables(2).ValuesAsNumpy()

#     # print(site_code, temperature[:5])

In [ ]:
# df = pd.DataFrame(
#         {
#             "site_code": site_code,
#             "time_interval": pd.date_range(
#                 start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
#                 end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
#                 freq=pd.Timedelta(seconds=hourly.Interval()),
#                 inclusive="right",
#             ),
#             "temperature": temperature,
#             "humidity": humidity,
#             "solar_radiance": radiation,
#         }
#     )
# df.to_csv("weather_data.csv", index=False)

In [ ]:
# def create_db_site_weather(db_name):
#     try:
#         with psycopg.connect(
#             dbname="csv_database",
#             user="postgres",
#             password="123789",
#             host="localhost",
#             port="5000",
#             autocommit=True,
#         ) as conn:
#             with conn.cursor() as cur:
#                 query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
#                 cur.execute(query)
#                 print(f"Database '{db_name}' created successfully!")

#     except psycopg.Error as e:
#         print(f"An error occurred: {e}")

In [ ]:
# create_db_site_weather("site_weather")

Database 'site_weather' created successfully!


In [ ]:
# def create_table_site_weather():
#     try:
#         with psycopg.connect(
#             dbname="site_weather",
#             user="postgres",
#             password="123789",
#             host="localhost",
#             port="5000",
#         ) as conn:

#             with conn.cursor() as cur:
#                 cur.execute("""
#                     CREATE TABLE IF NOT EXISTS site_weather (
#                         site_name VARCHAR(100) Not NULL,
#                         time_interval TIMESTAMPTZ Not NULL,
#                         temperature REAL Not NULL,
#                         humidity REAL Not NULL,
#                         solar_radiance REAL Not NULL
#                     );
#                 """)

#             conn.commit()
#             print("site_weather table created.")

#     except psycopg.Error as e:
#         print(e)

In [ ]:
# create_table_site_weather()

site_weather table created.


In [ ]:
# def data_store(sites_df):

#     with psycopg.connect(
#         dbname="site_weather",
#         user="postgres",
#         password="123789",
#         host="localhost",
#         port="5000",
#     ) as conn:
#         with conn.cursor() as cur:

#             for _, row in sites_df.iterrows():

#                 cur.execute(
#                     """
#                     INSERT INTO site_weather
#                     (site_name, time_interval, temperature, humidity, solar_radiance)
#                     VALUES (%s, %s, %s, %s, %s)
#                     """,
#                     (
#                         row["site_code"],
#                         row["time_interval"],
#                         row["temperature"],
#                         row["humidity"],
#                         row["solar_radiance"],
#                     ),
#                 )

#         conn.commit()

#     print("Site weather data inserted successfully!")

In [ ]:
# data_store(df)

Site weather data inserted successfully!
